## Phase 2:
- first preprocessing (loading and preparing the images).
- A dummy classifier that always says "normal" — and why its accuracy is misleading.
- A first real baseline model, simple, with its score.
- The metric that will be used for the whole project, with reasoning.

Important: 
One trap to avoid: never pick your decision threshold using the test set.

## Steps

#### Step 1: imports + Checking All 15 categories loading correctly. 

In [510]:
# (imports + categories)

from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image # (Pillow) is a Python library for opening, reading, and manipulating image files
from sklearn.metrics import average_precision_score, roc_auc_score, accuracy_score

DATA_ROOT = Path("../../data/raw/mvtec_ad")
categories = sorted([d.name for d in DATA_ROOT.iterdir() if d.is_dir() and (d / "test").exists()])
print(categories)

['bottle', 'cable', 'capsule', 'carpet', 'grid', 'hazelnut', 'leather', 'metal_nut', 'pill', 'screw', 'tile', 'toothbrush', 'transistor', 'wood', 'zipper']


#### Step 2: scope and constants:

In [511]:

# scope decision (which categories we are actually testing) and a couple of constants, SIZE, RANDOM_SEED, etc.

SIZE = 256  # working resolution, square
RANDOM_SEED = 42
VAL_FRACTION = 0.15  # slice of train_good held out ONLY for picking a threshold later

PHASE2_CATEGORIES = ["screw", "metal_nut", "tile", "grid"]
GRAYSCALE_CATEGORIES = {"grid", "screw", "tile", "toothbrush", "zipper"}

Why these values: 
- SIZE = 256: 
it is the pixel resolution that will be used to resize every image to, so they're all directly comparable (raw images vary between 700–1024 pixels, too big and inconsistent to compare pixel-by-pixel as-is). 
- PHASE2_CATEGORIES: 
it is the 4-category scope from EDA, screw and metal_nut are the extremes of pixel-level imbalance (0.34% vs 14.5%, the biggest split was  found) and tile and grid add two texture categories with different defect shapes. 
- GRAYSCALE_CATEGORIES:
it matches what EDA already confirmed, those 5 categories are natively grayscale, not RGB, so we shouldn't be force to convert them later.
- RANDOM_SEED = 42: 
when we split train_good into "fit" (used to build the model) and "val" (held back purely to pick a detection threshold), we do that split by shuffling the list of images into random order and cutting it. Without a fixed seed, "random" really means different every single time you re-run the notebook, so today's shuffle would differ from tomorrow's, and your fit/val split would silently change between runs.
That's a problem because it breaks reproducibility: if you show your baseline's score to others, then re-run the notebook tomorrow and get a slightly different score, nobody can tell if that's due to a real code change or just random luck from a different split. Setting RANDOM_SEED = 42 forces the "random" shuffle to always come out in the same order every time, same split, same results, every run, on any machine. The number 42 itself is arbitrary (it's a common reference to "The Hitchhiker's Guide to the Galaxy," where 42 is the supercomputer Deep Thought's answer to "the ultimate question of life, the universe, and everything."). So any other fixed number would work identically.
- VAL_FRACTION = 0.15:
Remember we can't pick our detection threshold using the test set, since that would mean cheating by peeking at the data we are supposed to be evaluated on. So we need a separate small slice of normal images, held out purely for choosing the threshold, that's neither used to build the model nor used for final scoring. That's what VAL_FRACTION carves out — 15% of the train_good images get set aside as this "val" set, the other 85% ("fit") is what actually builds the model.
Why 15% specifically: in our first report, we already found that each category only has roughly 200–300 normal training images total (look at Appendix E). Take screw, for example — it has around 320 training images. 15% of that is about 48 images held out for val, leaving around 270 to actually fit the model. That's enough images in val to get a stable estimate of what percentile of anomaly scores looks normal without cutting so deep into the fit set that the model has too little data left to learn from.
10% or 20% would also be defensible, but 15% is a reasonable middle ground given how small these training sets already are.

#### Step 3: functions to load one image and one mask 

In [512]:
# functions to load one image and one mask (load_image, load_mask)

def load_image(path, size, category):
    mode = "L" if category in GRAYSCALE_CATEGORIES else "RGB"
    img = Image.open(path).convert(mode).resize((size, size), Image.Resampling.BILINEAR)
    return np.asarray(img, dtype=np.float32) / 255.0


def load_mask(path, size):
    img = Image.open(path).convert("L").resize((size, size), Image.Resampling.NEAREST)
    arr = np.asarray(img, dtype=np.float32) / 255.0
    return (arr > 0.5).astype(np.float32)

#### Step 4: collecting file paths for a category

In [513]:
# collecting file paths for a category (load_category_paths)

def load_category_paths(category):
    cat_dir = DATA_ROOT / category
    train_good = sorted((cat_dir / "train" / "good").glob("*.png"))
    test_good = sorted((cat_dir / "test" / "good").glob("*.png"))

    test_defect, test_defect_masks = [], []
    for defect_dir in sorted((cat_dir / "test").iterdir()):
        if not defect_dir.is_dir() or defect_dir.name == "good":
            continue
        for img_path in sorted(defect_dir.glob("*.png")):
            mask_path = cat_dir / "ground_truth" / defect_dir.name / f"{img_path.stem}_mask.png"
            test_defect.append(img_path)
            test_defect_masks.append(mask_path)

    return train_good, test_good, test_defect, test_defect_masks

What it does: 
for one category (it could be "screw"), collects four lists of file paths:
- train_good: every normal training image (the only thing models get to learn from).
- test_good: normal images in the test set (used to check for false alarms).
- test_defect:  every defective test image, across all defect subtypes (e.g. screw/test/scratch_head/, screw/test/thread_top/, ..., the loop walks through each defect-type subfolder).
- test_defect_masks: the matching ground-truth mask path for each defective image, built by swapping the folder for ground_truth/<defect_type>/ and appending _mask to the filename — this mirrors exactly how MVTec AD names its mask files.

#### Quick check on "screw" category

In [514]:
# Quick check: file paths for one category (screw)

train_good, test_good, test_defect, test_defect_masks = load_category_paths("screw")
print(f"train_good={len(train_good)}  test_good={len(test_good)}  test_defect={len(test_defect)}")

train_good=320  test_good=41  test_defect=119


What this check tells: it calls the function on "screw" specifically and prints how many images landed in each list. We should see numbers matching what our EDA already found for screw, around 320 training images, 41 normal test images, and 119 defective test images (per our Report 1 Appendix E in our dataset composition table). If the numbers come out different or it errors, that tells us something is off in the path-matching logic before we move any further.

#### Step 6: the fit/val split function: splitting train_good into the 85% "fit" and 15% "val"

In [515]:
# the fit/val split function: splitting train_good into the 85% "fit" and 15% "val"

def split_fit_val(train_good, val_fraction=VAL_FRACTION, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    paths = list(train_good)
    rng.shuffle(paths)
    n_val = max(1, round(len(paths) * val_fraction))
    return paths[n_val:], paths[:n_val]  # fit, val

What it does: "np.random.default_rng(seed)" creates a random-number generator locked to the fixed seed (42), this is what makes the random shuffle reproducible every time it is re-run. "rng.shuffle(paths)" randomly reorders the list of image paths in place. "n_val" calculates how many images 15% represents, it rounded with max(1, ...) as a safety net so it never get zero validation images even for a tiny category. Then it slices the shuffled list into two pieces: everything after the first "n_val" images becomes "fit", the first "n_val" become "val".

#### Quick check: fit/val split

In [516]:
# Quick check: fit/val split

fit_paths, val_paths = split_fit_val(train_good)
print(f"fit={len(fit_paths)}  val={len(val_paths)}")

fit=272  val=48


It shows fit=272 val=48 (320 total, 15% of 320 rounds to 48)

#### Step 7: batch image loading

In [517]:
# batch image loading:

def load_stack(paths, size, category):
    return np.stack([load_image(p, size, category) for p in paths], axis=0)

Pay attention to what it does: takes a list of file paths, calls load_image function (from Step 3) on each one, and stacks all the resulting arrays together into a single NumPy array with one extra dimension on the front — so instead of 272 separate 256×256 arrays, we get one array of shape (272, 256, 256) (or (272, 256, 256, 3) for color categories). This is the format the dummy classifier and baseline will expect.

test only on val since it's small (48 images)and not all 272 fit images.

In [518]:
# test only on val 

val_images = load_stack(val_paths, SIZE, "screw")
print(val_images.shape)

(48, 256, 256)


(48, 256, 256) with no third channel dimension confirms grayscale handling is working (screw is one of your grayscale categories), and 48 matches the val count.

#### Step 8: the dummy classifier (it always predicts "normal")

In [519]:
# the dummy classifier: always predicts "normal"

def dummy_score(images):
    """Assigns every pixel an anomaly score of 0 -- never flags anything as anomalous."""
    if images.ndim == 4:  # color images: (N, H, W, C)
        return np.zeros(images.shape[:3], dtype=np.float32)
    return np.zeros(images.shape, dtype=np.float32)  # grayscale: (N, H, W)

What it does: no learning, no parameters, no looking at any data. Every pixel of every image gets a score of exactly 0, meaning "not anomalous." It's the simplest possible baseline, not a real candidate model.

Then use the dummy classifier on real data to see the accuracy paradox appear in our own numbers.

In [520]:
# Quick check: loading test images and masks for "screw"

test_good_images = load_stack(test_good, SIZE, "screw")
test_defect_images = load_stack(test_defect, SIZE, "screw")
test_defect_mask_arrays = np.stack([load_mask(p, SIZE) for p in test_defect_masks], axis=0)

print(f"test_good: {test_good_images.shape}  test_defect: {test_defect_images.shape}  masks: {test_defect_mask_arrays.shape}")

test_good: (41, 256, 256)  test_defect: (119, 256, 256)  masks: (119, 256, 256)


For the next stepcomputing the dummy classifier's accuracy vs its real score.

#### Step 9: dummy classifier metrics
Dummy classifier: accuracy vs. Average Precision

Score the "always normal" classifier on real screw test data. 
Dummy classifier always returns 0.

In [521]:
dummy_good_scores = dummy_score(test_good_images)
dummy_defect_scores = dummy_score(test_defect_images)

Flatten everything into two matching 1D lists: one true label per pixel, one predicted score per pixel, in the same order.

In [522]:
y_true = np.concatenate([
    np.zeros(dummy_good_scores.size, dtype=np.float32),
    test_defect_mask_arrays.reshape(-1),
])
y_score = np.concatenate([
    dummy_good_scores.reshape(-1),
    dummy_defect_scores.reshape(-1),
])

Comparing the naive accuracy (predict normal everywhere) against the honest score, Average Precision.

In [523]:
naive_accuracy = 1.0 - y_true.mean()

y_true.mean() is the average of a bunch of 0s and 1s, which equals the fraction of pixels that are truly anomalous. Since the dummy predicts "normal" for literally every pixel, it's correct everywhere except the truly anomalous ones, so its accuracy is just "1 minus the fraction that are actually anomalous."

In [524]:
dummy_ap = average_precision_score(y_true, y_score)

This uses sklearn's built-in Average Precision function, comparing  true labels against the predicted scores. 
why this comes out so close to the anomaly rate here specifically: since every single prediction is tied at exactly 0 (no variation at all), there's no way to rank pixels as "more or less suspicious" so sklearn essentially can't distinguish anything and the score goes down to almost the base rate of positives in the data. 
That's mathematically why dummy_ap, matching almost exactly the true anomalous-pixel fraction.

In [525]:
print(f"Naive pixel accuracy (predict normal everywhere): {naive_accuracy:.4f}")
print(f"Pixel Average Precision: {dummy_ap:.4f}")

Naive pixel accuracy (predict normal everywhere): 0.9975
Pixel Average Precision: 0.0025


This is the accuracy paradox, proven in our own screw data. A classifier that does absolutely nothing, no learning, no logic and just predict normal everywhere and it looks like a near-perfect model by accuracy (99.75%!), while its actual, honest score (Average Precision) is essentially zero. It's not really 99.75% good, this accuracy just hides that fact because normal pixels so overwhelmingly dominate the count. 

That gap between 0.9975 and 0.0025 is our answer to the question, what it shows about its accuracy, it means nothing, it's an imbalance and not a sign of a working model.

remmember that our EDA found screw's anomalous-pixel ratio at ~0.34% and this run gives 0.25%, it is close but not identical, which makes sense since this includes the 41 normal test images too, which drags the overall anomaly ratio down slightly further than the defective-images-only number in our report. I is just a different denominator.